# Agent Identity SDK: Utility & Cleanup Tools

This notebook provides helper functions to manage and clean up resources created while testing the `agent-identity-dev-sdk`. It demonstrates how to interact **directly** with the underlying `huaweicloudsdkagentidentity` SDK for administrative tasks like resource cleanup and configuration updates.

### Prerequisites
Ensure you have set your **Huawei Cloud AK/SK** credentials and the desired region.

In [ ]:
import os
from typing import Optional, List
from agentarts.sdk import IdentityClient
from huaweicloudsdkagentidentity.v1 import (
    ListWorkloadIdentitiesRequest,
    GetWorkloadIdentityRequest,
    DeleteWorkloadIdentityRequest,
    UpdateWorkloadIdentityRequest,
    UpdateWorkloadIdentityReqBody,
    ListOauth2CredentialProvidersRequest,
    GetOauth2CredentialProviderRequest,
    DeleteOauth2CredentialProviderRequest,
    UpdateOauth2CredentialProviderRequest,
    UpdateOauth2CredentialProviderReqBody,
    Oauth2ProviderConfigInput,
    GoogleOauth2ProviderConfigInput,
    ListApiKeyCredentialProvidersRequest,
    GetApiKeyCredentialProviderRequest,
    DeleteApiKeyCredentialProviderRequest,
    ListStsCredentialProvidersRequest,
    GetStsCredentialProviderRequest,
    DeleteStsCredentialProviderRequest,
)

# Initialize the IdentityClient from the SDK to get the underlying raw client
region = os.getenv("HUAWEICLOUD_SDK_REGION", "ap-southeast-4")
client = IdentityClient(region=region, ignore_ssl_verification=True)
raw_client = client.client  # Access the underlying AgentIdentityClient

print(f"Utility tools initialized for region: {region}")

## 1. Cleanup Tools

These functions help you find and delete resources created during experimentation. They use the `raw_client` to invoke the Identity service APIs directly.

In [ ]:
def cleanup_workload_identities(prefix: Optional[str] = None):
    """List and delete workload identities, optionally filtered by prefix."""
    print(f"Listing workload identities (prefix='{prefix or ''}')...")
    response = raw_client.list_workload_identities(ListWorkloadIdentitiesRequest())
    workloads = response.workload_identities or []

    deleted_count = 0
    for wl in workloads:
        if not prefix or wl.name.startswith(prefix):
            print(f"  Deleting Workload Identity: {wl.name}")
            raw_client.delete_workload_identity(DeleteWorkloadIdentityRequest(workload_identity_name=wl.name))
            deleted_count += 1

    print(f"Cleanup complete. Deleted {deleted_count} workload(s).")


def cleanup_oauth2_providers(prefix: Optional[str] = None):
    """List and delete OAuth2 credential providers."""
    print(f"Listing OAuth2 providers (prefix='{prefix or ''}')...")
    response = raw_client.list_oauth2_credential_providers(ListOauth2CredentialProvidersRequest())
    providers = response.credential_providers or []

    deleted_count = 0
    for p in providers:
        if not prefix or p.name.startswith(prefix):
            print(f"  Deleting OAuth2 Provider: {p.name}")
            raw_client.delete_oauth2_credential_provider(
                DeleteOauth2CredentialProviderRequest(credential_provider_name=p.name))
            deleted_count += 1

    print(f"Cleanup complete. Deleted {deleted_count} provider(s).")


def cleanup_api_key_providers(prefix: Optional[str] = None):
    """List and delete API Key credential providers."""
    print(f"Listing API Key providers (prefix='{prefix or ''}')...")
    response = raw_client.list_api_key_credential_providers(ListApiKeyCredentialProvidersRequest())
    providers = response.credential_providers or []

    deleted_count = 0
    for p in providers:
        if not prefix or p.name.startswith(prefix):
            print(f"  Deleting API Key Provider: {p.name}")
            raw_client.delete_api_key_credential_provider(
                DeleteApiKeyCredentialProviderRequest(credential_provider_name=p.name))
            deleted_count += 1

    print(f"Cleanup complete. Deleted {deleted_count} provider(s).")


def cleanup_sts_providers(prefix: Optional[str] = None):
    """List and delete STS credential providers."""
    print(f"Listing STS providers (prefix='{prefix or ''}')...")
    response = raw_client.list_sts_credential_providers(ListStsCredentialProvidersRequest())
    providers = response.credential_providers or []

    deleted_count = 0
    for p in providers:
        if not prefix or p.name.startswith(prefix):
            print(f"  Deleting STS Provider: {p.name}")
            raw_client.delete_sts_credential_provider(
                DeleteStsCredentialProviderRequest(credential_provider_name=p.name))
            deleted_count += 1

    print(f"Cleanup complete. Deleted {deleted_count} provider(s).")


print("Cleanup utility functions defined.")

### Run Cleanup

Uncomment the lines below to perform cleanup for specific test patterns.

In [ ]:
# cleanup_workload_identities(prefix="oauth-workload-")
# cleanup_oauth2_providers(prefix="google-oauth-")
# cleanup_api_key_providers(prefix="my-llm-provider-")
# cleanup_sts_providers(prefix="my-sts-provider-")

## 2. Get Tools

These functions help you retrieve detailed information about specific resources.

In [ ]:
def get_workload_identity(workload_name: str):
    """Retrieve details for a specific workload identity."""
    print(f"Getting Workload Identity: {workload_name}...")
    request = GetWorkloadIdentityRequest(workload_identity_name=workload_name)
    response = raw_client.get_workload_identity(request)
    return response.workload_identity


def get_oauth2_provider(provider_name: str):
    """Retrieve details for a specific OAuth2 credential provider."""
    print(f"Getting OAuth2 Provider: {provider_name}...")
    request = GetOauth2CredentialProviderRequest(credential_provider_name=provider_name)
    response = raw_client.get_oauth2_credential_provider(request)
    return response.credential_provider


def get_api_key_provider(provider_name: str):
    """Retrieve details for a specific API Key credential provider."""
    print(f"Getting API Key Provider: {provider_name}...")
    request = GetApiKeyCredentialProviderRequest(credential_provider_name=provider_name)
    response = raw_client.get_api_key_credential_provider(request)
    return response.credential_provider


def get_sts_provider(provider_name: str):
    """Retrieve details for a specific STS credential provider."""
    print(f"Getting STS Provider: {provider_name}...")
    request = GetStsCredentialProviderRequest(credential_provider_name=provider_name)
    response = raw_client.get_sts_credential_provider(request)
    return response.credential_provider


print("Get utility functions defined.")

### Run Get

Uncomment the lines below to retrieve specific resources.

In [ ]:
# wl = get_workload_identity("my-workload-name")
# print(wl)
# op = get_oauth2_provider("my-oauth2-provider")
# print(op)
# ap = get_api_key_provider("my-api-key-provider")
# print(ap)
# sp = get_sts_provider("my-sts-provider")
# print(sp)

## 3. Update Tools

This section demonstrates how to use the underlying SDK models to update existing credential providers.

In [ ]:
def update_google_oauth2_secret(provider_name: str, client_id: str, new_client_secret: str):
    """Demonstrates updating the client secret for an existing Google OAuth2 provider using raw SDK models."""
    print(f"Updating OAuth2 provider '{provider_name}' with new secret...")

    # We construct the update request using the raw SDK models from huaweicloudsdkagentidentity
    update_body = UpdateOauth2CredentialProviderReqBody(
        oauth2_provider_config_input=Oauth2ProviderConfigInput(
            google_oauth2_provider_config=GoogleOauth2ProviderConfigInput(
                client_id=client_id,
                client_secret=new_client_secret
            )
        )
    )

    request = UpdateOauth2CredentialProviderRequest(
        credential_provider_name=provider_name,
        body=update_body
    )

    # Invoke the update via the raw AgentIdentityClient
    response = raw_client.update_oauth2_credential_provider(request)
    print(f"Successfully updated provider: {response.credential_provider.name}")
    return response.credential_provider


def update_workload_identity_urls(workload_name: str, allowed_urls: List[str]):
    """Update the allowed OAuth2 return URLs for a workload identity using raw SDK models."""
    print(f"Updating workload identity '{workload_name}' with allowed URLs: {allowed_urls}...")

    # Construct the update request using the raw SDK models
    update_body = UpdateWorkloadIdentityReqBody(
        allowed_resource_oauth2_return_urls=allowed_urls
    )

    request = UpdateWorkloadIdentityRequest(
        workload_identity_name=workload_name,
        body=update_body
    )

    # Invoke the update via the raw AgentIdentityClient
    response = raw_client.update_workload_identity(request)
    print(f"Successfully updated workload identity: {response.workload_identity.name}")
    return response.workload_identity


print("Update utility functions defined.")

### Run Updates

Use the cells below to manually update existing resources. You can run these independently.

In [ ]:
# test_provider_name = "test-update-p-xxxx"
# print(f"Updating provider {test_provider_name}...")
# update_google_oauth2_secret(
#     provider_name=test_provider_name,
#     client_id="initial-client-id",
#     new_client_secret="newly-rotated-secret"
# )

In [ ]:
# test_workload_name = "test-update-w-xxxx"
# print(f"Updating workload identity {test_workload_name}...")
# # Update to allow multiple URLs
# update_workload_identity_urls(
#     workload_name=test_workload_name,
#     allowed_urls=["http://localhost:8000/callback", "https://myapp.com/callback"]
# )